<a href="https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/inspect_data1.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# MarinFold Data Inspector

This notebook is for browsing MarinFold training data in Colab. It covers two source families:

- the legacy `timodonnell/protein-docs` Hugging Face datasets
- the newer `open-athena/MarinFold` bucket data under `data/document_structures/`

The legacy side is streamed directly from the Hub. The bucket side does a cheap live file listing first, then downloads just one selected parquet file for schema and sample-row inspection.


In [ ]:
%pip install -q "datasets>=3.1,<4" "huggingface_hub>=1.5" pyarrow pandas


In [ ]:
import os
from collections import defaultdict
from itertools import islice
from pathlib import Path, PurePosixPath

import pandas as pd
import pyarrow.parquet as pq
from datasets import get_dataset_config_names, load_dataset, load_dataset_builder
from huggingface_hub import HfApi
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 160)

# @title Configuration
LEGACY_DATASET = "timodonnell/protein-docs"
LEGACY_SPLIT = "train"  # @param ["train", "val", "test"]
LEGACY_SUBSETS_CSV = "contacts-and-distances-v1-5x, deterministic-positives-only, random-3-bins"  # @param {type:"string"}
BUCKET_ID = "open-athena/MarinFold"
BUCKET_PREFIX = "data/document_structures/"  # @param {type:"string"}
BUCKET_COLLECTION_PREFIX = "data/document_structures/contacts_v1/train"  # @param {type:"string"}
BUCKET_FILE_INDEX = 0  # @param {type:"integer"}
N_EXAMPLES = 3  # @param {type:"integer"}
DOC_PREVIEW_CHARS = 1200
MAX_BUCKET_COLLECTIONS_TO_SHOW = 30

LEGACY_SUBSETS = [x.strip() for x in LEGACY_SUBSETS_CSV.split(",") if x.strip()]
COLAB_ROOT = Path("/content")
BASE_WORKDIR = COLAB_ROOT if COLAB_ROOT.exists() and os.access(COLAB_ROOT, os.W_OK) else Path.cwd()
CACHE_DIR = BASE_WORKDIR / "marinfold_data_inspector_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(
    {
        "legacy_dataset": LEGACY_DATASET,
        "legacy_split": LEGACY_SPLIT,
        "legacy_subsets": LEGACY_SUBSETS,
        "bucket_id": BUCKET_ID,
        "bucket_prefix": BUCKET_PREFIX,
        "bucket_collection_prefix": BUCKET_COLLECTION_PREFIX,
        "bucket_file_index": BUCKET_FILE_INDEX,
        "n_examples": N_EXAMPLES,
    }
)


In [ ]:
def format_bytes(num_bytes: int) -> str:
    value = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if value < 1024 or unit == "TB":
            return f"{value:.1f} {unit}"
        value /= 1024
    return f"{value:.1f} TB"


def truncate_text(value, max_chars: int = 160) -> str:
    text = str(value)
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 3] + "..."


def pick_document_column(columns):
    preferred = ["document", "text", "doc"]
    for name in preferred:
        if name in columns:
            return name
    for name in columns:
        if isinstance(name, str) and ("document" in name or "text" in name):
            return name
    return None


def preview_dataframe(rows, *, preview_chars: int = 160):
    if not rows:
        return pd.DataFrame(), None
    doc_col = pick_document_column(rows[0].keys())
    cooked = []
    for row in rows:
        out = {}
        for key, value in row.items():
            if key == doc_col:
                out[f"{key}_preview"] = truncate_text(value, preview_chars)
            elif isinstance(value, str):
                out[key] = truncate_text(value, preview_chars)
            else:
                out[key] = value
        cooked.append(out)
    return pd.DataFrame(cooked), doc_col


def show_document_samples(rows, doc_col, *, title: str) -> None:
    if doc_col is None:
        print(f"{title}: no obvious document/text column found.")
        return
    display(Markdown(f"### {title}"))
    for idx, row in enumerate(rows, start=1):
        sample = truncate_text(row[doc_col], DOC_PREVIEW_CHARS)
        print(f"Sample {idx} ({doc_col}):\n")
        print(sample)
        print("\n" + "-" * 100 + "\n")


def schema_dataframe(schema) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "column": field.name,
                "type": str(field.type),
                "nullable": field.nullable,
            }
            for field in schema
        ]
    )


## Legacy `protein-docs` datasets

This section queries the published `timodonnell/protein-docs` dataset repo directly from the Hub and streams a few examples from the selected subsets.


In [ ]:
available_legacy_configs = get_dataset_config_names(LEGACY_DATASET)
display(pd.DataFrame({"available_subset": available_legacy_configs}))

selected_legacy_subsets = [name for name in LEGACY_SUBSETS if name in available_legacy_configs]
missing_legacy_subsets = [name for name in LEGACY_SUBSETS if name not in available_legacy_configs]

print("Selected subsets:", selected_legacy_subsets)
if missing_legacy_subsets:
    print("Missing subsets:", missing_legacy_subsets)


In [ ]:
legacy_summary_rows = []
legacy_samples = {}

for subset in selected_legacy_subsets:
    builder = load_dataset_builder(LEGACY_DATASET, subset)
    features = builder.info.features or {}
    split_info = (builder.info.splits or {}).get(LEGACY_SPLIT)
    row_count = getattr(split_info, "num_examples", None)
    legacy_summary_rows.append(
        {
            "subset": subset,
            "split": LEGACY_SPLIT,
            "num_examples": row_count,
            "columns": list(features.keys()),
        }
    )
    ds = load_dataset(LEGACY_DATASET, subset, split=LEGACY_SPLIT, streaming=True)
    legacy_samples[subset] = list(islice(ds, N_EXAMPLES))

legacy_summary_df = pd.DataFrame(legacy_summary_rows)
display(legacy_summary_df)

for subset, rows in legacy_samples.items():
    display(Markdown(f"### Legacy subset: `{subset}`"))
    preview_df, doc_col = preview_dataframe(rows)
    display(preview_df)
    show_document_samples(rows, doc_col, title=f"Legacy documents from `{subset}`")


## Live bucket data

This section uses the Hugging Face bucket API to list current files in `open-athena/MarinFold`. Listing is cheap and live. For parquet schema/sample inspection, the notebook downloads only the single selected parquet file into the Colab runtime cache.


In [ ]:
api = HfApi()
bucket_info = api.bucket_info(BUCKET_ID)
display(
    pd.DataFrame(
        [
            {
                "bucket_id": bucket_info.id,
                "private": bucket_info.private,
                "total_files": bucket_info.total_files,
                "bucket_size": format_bytes(bucket_info.size),
            }
        ]
    )
)

bucket_items = sorted(api.list_bucket_tree(BUCKET_ID, recursive=True), key=lambda item: item.path)
parquet_items = [
    item
    for item in bucket_items
    if getattr(item, "type", None) != "directory"
    and item.path.startswith(BUCKET_PREFIX)
    and item.path.endswith(".parquet")
]

collection_summary = defaultdict(lambda: {"parquet_files": 0, "bytes": 0, "sample_file": None})
for item in parquet_items:
    collection = str(PurePosixPath(item.path).parent)
    collection_summary[collection]["parquet_files"] += 1
    collection_summary[collection]["bytes"] += int(getattr(item, "size", 0) or 0)
    collection_summary[collection]["sample_file"] = collection_summary[collection]["sample_file"] or item.path

bucket_summary_df = (
    pd.DataFrame(
        [
            {
                "collection": collection,
                "parquet_files": values["parquet_files"],
                "total_size": format_bytes(values["bytes"]),
                "sample_file": values["sample_file"],
            }
            for collection, values in collection_summary.items()
        ]
    )
    .sort_values(["parquet_files", "collection"], ascending=[False, True])
    .reset_index(drop=True)
)

display(bucket_summary_df.head(MAX_BUCKET_COLLECTIONS_TO_SHOW))
print(f"Matched parquet files under {BUCKET_PREFIX!r}: {len(parquet_items):,}")


In [ ]:
selected_bucket_files = [
    item for item in parquet_items if item.path.startswith(BUCKET_COLLECTION_PREFIX)
]
selected_bucket_files = sorted(selected_bucket_files, key=lambda item: item.path)

if not selected_bucket_files:
    raise ValueError(
        f"No parquet files found under {BUCKET_COLLECTION_PREFIX!r}. "
        f"Try a different BUCKET_COLLECTION_PREFIX from the summary table above."
    )
if BUCKET_FILE_INDEX < 0 or BUCKET_FILE_INDEX >= len(selected_bucket_files):
    raise IndexError(
        f"BUCKET_FILE_INDEX={BUCKET_FILE_INDEX} is out of range for "
        f"{len(selected_bucket_files)} files under {BUCKET_COLLECTION_PREFIX!r}."
    )

selected_bucket_item = selected_bucket_files[BUCKET_FILE_INDEX]
selected_bucket_path = selected_bucket_item.path
selected_bucket_size = int(getattr(selected_bucket_item, "size", 0) or 0)

display(
    pd.DataFrame(
        [
            {
                "selected_bucket_path": selected_bucket_path,
                "size": format_bytes(selected_bucket_size),
                "file_index": BUCKET_FILE_INDEX,
                "files_in_collection": len(selected_bucket_files),
            }
        ]
    )
)


In [ ]:
bucket_cache_dir = CACHE_DIR / "bucket_parquet"
bucket_cache_dir.mkdir(parents=True, exist_ok=True)
local_bucket_path = bucket_cache_dir / PurePosixPath(selected_bucket_path).name

api.download_bucket_files(BUCKET_ID, files=[(selected_bucket_path, str(local_bucket_path))])
parquet_file = pq.ParquetFile(local_bucket_path)

schema_df = schema_dataframe(parquet_file.schema_arrow)
display(schema_df)

file_metadata = parquet_file.metadata
display(
    pd.DataFrame(
        [
            {
                "num_rows": file_metadata.num_rows,
                "num_row_groups": file_metadata.num_row_groups,
                "serialized_size": format_bytes(file_metadata.serialized_size),
                "created_by": file_metadata.created_by,
            }
        ]
    )
)

key_value_metadata = file_metadata.metadata or {}
if key_value_metadata:
    metadata_df = pd.DataFrame(
        [
            {
                "key": key.decode() if isinstance(key, bytes) else str(key),
                "value": truncate_text(
                    value.decode() if isinstance(value, bytes) else str(value),
                    200,
                ),
            }
            for key, value in key_value_metadata.items()
        ]
    )
    display(metadata_df)
else:
    print("No parquet key-value metadata found on this file.")


In [ ]:
preview_table = parquet_file.read_row_group(0)
preview_rows = preview_table.slice(0, N_EXAMPLES).to_pylist()
bucket_preview_df, bucket_doc_col = preview_dataframe(preview_rows)
display(bucket_preview_df)
show_document_samples(
    preview_rows,
    bucket_doc_col,
    title=f"Bucket documents from `{selected_bucket_path}`",
)
